In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# ── Color palette ──────────────────────────────────────────────────────────────
COLOR_A = "#4C72B0"   # blue  – Version A
COLOR_B = "#DD8452"   # orange – Version B

# 1. DATA LOADING & CLEANING
df_raw = pd.read_csv("quiz_responses.csv")
print(f"Raw data shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")

# 1.1 Parse timestamps
df = df_raw.copy()

correct = {
    "q1": "Dolphin",   # mammal
    "q2": "Asia",      # largest continent
    "q3": "Water",     # NOT a gas at room temp
    "q4": "1/3",       # P(>4 on d6) = 2/6 = 1/3
    "q5": 4,           # NOT prime
}

for q, ans in correct.items():
    df[f"{q}_correct"] = (df[q].astype(str).str.strip() == str(ans)).astype(int)

q1_dur = df["duration_sec"].quantile(0.25)
q3_dur = df["duration_sec"].quantile(0.75)
iqr    = q3_dur - q1_dur
lower_bound = max(0, q1_dur - 3 * iqr)
upper_bound = q3_dur + 3 * iqr
outliers = df[(df["duration_sec"] < lower_bound) | (df["duration_sec"] > upper_bound)]
print(f"\nDuration outliers (3×IQR rule): {len(outliers)} rows")
print(f"  Lower bound: {lower_bound:.1f}s  |  Upper bound: {upper_bound:.1f}s")
if len(outliers) > 0:
    print(outliers[["id", "group_name", "duration_sec"]])

Raw data shape: (59, 16)
Columns: ['id', 'user_id', 'group_name', 'start_time', 'submit_time', 'duration_sec', 'score', 'completed', 'satisfaction', 'ease_of_use', 'q1', 'q2', 'q3', 'q4', 'q5', 'created_at']

Duration outliers (3×IQR rule): 1 rows
  Lower bound: 0.0s  |  Upper bound: 314.0s
    id group_name  duration_sec
46  53          A    340.439995


In [2]:
# Remove extreme outliers for duration analysis (keep for score/satisfaction)
df_clean = df[(df["duration_sec"] >= lower_bound) & (df["duration_sec"] <= upper_bound)].copy()
print(f"\nClean dataset for duration analysis: {len(df_clean)} rows")

A = df[df["group_name"] == "A"]
B = df[df["group_name"] == "B"]
A_c = df_clean[df_clean["group_name"] == "A"]
B_c = df_clean[df_clean["group_name"] == "B"]

print(f"\nGroup sizes after dedup: A={len(A)}, B={len(B)}")


Clean dataset for duration analysis: 58 rows

Group sizes after dedup: A=31, B=28


In [3]:
# 2. DESCRIPTIVE STATISTICS
metrics = ["score", "duration_sec", "satisfaction", "ease_of_use"]
desc = df.groupby("group_name")[metrics].agg(["mean", "median", "std"])
print(desc.round(3).to_string())

# Per-question accuracy
print("\nPer-question accuracy by group:")
q_acc = []
for q in ["q1","q2","q3","q4","q5"]:
    acc_a = A[f"{q}_correct"].mean()
    acc_b = B[f"{q}_correct"].mean()
    q_acc.append({"Question": q, "Version_A": round(acc_a,3), "Version_B": round(acc_b,3),
                  "Diff(B-A)": round(acc_b - acc_a, 3)})
q_acc_df = pd.DataFrame(q_acc)
print(q_acc_df.to_string(index=False))

            score               duration_sec                 satisfaction               ease_of_use              
             mean median    std         mean  median     std         mean median    std        mean median    std
group_name                                                                                                       
A           4.323    5.0  0.979       86.969  62.505  72.120        4.194    4.0  0.833       4.290    5.0  0.902
B           4.214    5.0  1.067       92.668  91.105  36.752        3.571    3.0  0.959       3.821    3.0  0.945

Per-question accuracy by group:
Question  Version_A  Version_B  Diff(B-A)
      q1      0.839      0.964      0.126
      q2      0.935      0.714     -0.221
      q3      0.903      0.857     -0.046
      q4      0.839      0.821     -0.017
      q5      0.806      0.857      0.051


In [9]:
# 3. STATISTICAL TESTS
alpha = 0.05
 
# t-test for score, satisfaction, ease_of_use
print(f"\n{'Metric':12} | {'T-stat':>8} | {'P-value':>8}")
print("-" * 38)
p_vals = {}
metrics_ttest = ['score', 'satisfaction', 'ease_of_use']
for metric in metrics_ttest:
    t_stat, p_val = stats.ttest_ind(A[metric], B[metric], equal_var=False)
    print(f"{metric:12} | T-stat: {t_stat:6.3f} | P-value: {p_val:6.4f} {'*' if p_val < 0.05 else ''}")
    p_vals[metric] = p_val
 
# Mann-Whitney for duration (skewed distribution)
u_stat, p_val_u = stats.mannwhitneyu(A_c["duration_sec"], B_c["duration_sec"])
print(f"{'duration_sec':12} | U-stat: {u_stat:6.1f} | P-value: {p_val_u:6.4f} {'*' if p_val_u < 0.05 else ''}")
p_vals["duration_sec"] = p_val_u


Metric       |   T-stat |  P-value
--------------------------------------
score        | T-stat:  0.405 | P-value: 0.6872 
satisfaction | T-stat:  2.646 | P-value: 0.0107 *
ease_of_use  | T-stat:  1.945 | P-value: 0.0568 
duration_sec | U-stat:  295.0 | P-value: 0.0526 


In [10]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle("A/B Test: Version A (All-at-Once) vs Version B (One-by-One with Feedback)",
             fontsize=12, fontweight="bold", y=1.01)
plt.subplots_adjust(hspace=0.45, wspace=0.35)
 
def add_pval(ax, p, x=0.5, y=0.97):
    label = f"p={p:.4f}{'*' if p < 0.05 else ''}"
    ax.text(x, y, label, transform=ax.transAxes, ha="center", va="top",
            fontsize=9, color="#c44e52" if p < 0.05 else "gray")
 
# ── Panel 1: Score distribution (boxplot + jitter) ───────────────────────────
ax1 = axes[0, 0]
groups_score = [A["score"].values, B["score"].values]
bp = ax1.boxplot(groups_score, patch_artist=True, widths=0.4,
                 medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp["boxes"], [COLOR_A, COLOR_B]):
    patch.set_facecolor(color); patch.set_alpha(0.65)
for i, (grp, col) in enumerate(zip(groups_score, [COLOR_A, COLOR_B]), 1):
    jitter = np.random.uniform(-0.12, 0.12, size=len(grp))
    ax1.scatter(np.full(len(grp), i) + jitter, grp, color=col, alpha=0.5, s=20, zorder=3)
ax1.set_xticks([1, 2]); ax1.set_xticklabels(["Version A", "Version B"])
ax1.set_ylabel("Score (out of 5)"); ax1.set_title("Score Distribution")
ax1.set_ylim(0.5, 6.0)
add_pval(ax1, p_vals["score"])
 
# ── Panel 2: Duration boxplot + jitter ───────────────────────────────────────
ax2 = axes[0, 1]
groups_dur = [A_c["duration_sec"].values, B_c["duration_sec"].values]
bp2 = ax2.boxplot(groups_dur, patch_artist=True, widths=0.4,
                  medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp2["boxes"], [COLOR_A, COLOR_B]):
    patch.set_facecolor(color); patch.set_alpha(0.65)
for i, (grp, col) in enumerate(zip(groups_dur, [COLOR_A, COLOR_B]), 1):
    jitter = np.random.uniform(-0.12, 0.12, size=len(grp))
    ax2.scatter(np.full(len(grp), i) + jitter, grp, color=col, alpha=0.5, s=20, zorder=3)
ax2.set_xticks([1, 2]); ax2.set_xticklabels(["Version A", "Version B"])
ax2.set_ylabel("Duration (seconds)"); ax2.set_title("Time to Complete Quiz")
add_pval(ax2, p_vals["duration_sec"])
 
# ── Panel 3: Satisfaction grouped bar ────────────────────────────────────────
ax3 = axes[1, 0]
cats    = ["Satisfaction", "Ease of Use"]
means_a = [A["satisfaction"].mean(), A["ease_of_use"].mean()]
means_b = [B["satisfaction"].mean(), B["ease_of_use"].mean()]
se_a    = [A["satisfaction"].sem(),  A["ease_of_use"].sem()]
se_b    = [B["satisfaction"].sem(),  B["ease_of_use"].sem()]
x = np.arange(len(cats)); width = 0.32
ax3.bar(x - width/2, means_a, width, color=COLOR_A, alpha=0.8, label="Version A", yerr=se_a, capsize=4)
ax3.bar(x + width/2, means_b, width, color=COLOR_B, alpha=0.8, label="Version B", yerr=se_b, capsize=4)
ax3.set_xticks(x); ax3.set_xticklabels(cats)
ax3.set_ylabel("Mean Rating (1–5)"); ax3.set_title("User Experience Ratings (Mean ± SE)")
ax3.set_ylim(0, 5.8); ax3.legend(fontsize=8)
# annotate p-values above each pair
for xi, metric_key in zip([0, 1], ["satisfaction", "ease_of_use"]):
    p = p_vals[metric_key]
    top = max(means_a[xi] + se_a[xi], means_b[xi] + se_b[xi]) + 0.25
    ax3.text(xi, top, f"p={p:.4f}{'*' if p < 0.05 else ''}",
             ha="center", fontsize=8, color="#c44e52" if p < 0.05 else "gray")
 
# ── Panel 4: Per-question accuracy ───────────────────────────────────────────
ax4 = axes[1, 1]
q_labels   = ["Q1\n(Mammal)", "Q2\n(Continent)", "Q3\n(Gas)", "Q4\n(Prob.)", "Q5\n(Prime)"]
acc_a_vals = [A[f"q{i}_correct"].mean() for i in range(1, 6)]
acc_b_vals = [B[f"q{i}_correct"].mean() for i in range(1, 6)]
x = np.arange(5); width = 0.35
bars_a = ax4.bar(x - width/2, acc_a_vals, width, color=COLOR_A, alpha=0.8, label="Version A")
bars_b = ax4.bar(x + width/2, acc_b_vals, width, color=COLOR_B, alpha=0.8, label="Version B")
ax4.set_xticks(x); ax4.set_xticklabels(q_labels)
ax4.set_ylabel("Accuracy"); ax4.set_title("Per-Question Accuracy by Version")
ax4.set_ylim(0, 1.2); ax4.legend(fontsize=8)
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
for bar in bars_a:
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
             f"{bar.get_height():.0%}", ha="center", va="bottom", fontsize=7, color=COLOR_A)
for bar in bars_b:
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
             f"{bar.get_height():.0%}", ha="center", va="bottom", fontsize=7, color=COLOR_B)
 
plt.savefig("ab_test_analysis.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: ab_test_analysis.png")

  Saved: ab_test_analysis.png
